In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
from pyspark.sql.functions import asc

In [0]:
#movie_df = spark.read.parquet(f"{silver_folder_path}/movies").filter("year_release_date >= 2010")
#movie_df = spark.read.table("movie_silver.movies").filter("year_release_date >= 2010")
movie_df = spark.read.table("movie_silver.movies").filter(f"file_date >= '{v_file_date}'")

In [0]:
#production_country_df = spark.read.parquet(f"{silver_folder_path}/production_country")
production_country_df = spark.read.table("movie_silver.production_countries").filter(f"file_date >= '{v_file_date}'")

In [0]:
#country_df = spark.read.parquet(f"{silver_folder_path}/countries")
country_df = spark.read.table("movie_silver.countries")

In [0]:
#movie_company_df = spark.read.parquet(f"{silver_folder_path}/movie_company")
movie_company_df = spark.read.table("movie_silver.movie_companies").filter(f"file_date >= '{v_file_date}'")

In [0]:
#production_company_df = spark.read.parquet(f"{silver_folder_path}/production_company")
production_company_df = spark.read.table("movie_silver.production_companies").filter(f"file_date >= '{v_file_date}'")

In [0]:
movie_final_df = movie_df \
            .join(production_country_df, movie_df.movie_id == production_country_df.movie_id, "inner") \
            .join(country_df, production_country_df.country_id == country_df.country_id, "inner") \
            .join(movie_company_df, movie_df.movie_id == movie_company_df.movie_id, "inner") \
            .join(production_company_df, movie_company_df.company_id == production_company_df.company_id, "inner") \
            .select (movie_df.movie_id, movie_df.title, movie_df.budget, movie_df.revenue, movie_df.duration_time, movie_df.release_date, country_df.country_name, production_company_df.company_name, production_company_df.company_id, production_country_df.country_id).orderBy(asc("title"))

In [0]:
movie_filter_df = movie_final_df.filter("year_release_date >= 2010")

In [0]:
from pyspark.sql.functions import lit

In [0]:
result_df = movie_filter_df.withColumn("created_date", lit(v_file_date))

In [0]:
display(result_df)

####Escribir los datos en el DataLake en formato "Parquet"

In [0]:
#overwrite_partition("movie_gold", "results_movies_production_company", "created_date", v_file_date)

In [0]:
#result_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_movies_production_company")
#result_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_movies_production_company")
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.company_id = src.company_id AND tgt.created_date = src.created_date'
merge_delta_lake (result_df, "movie_gold", "results_movies_production_company", merge_condition, "created_date")

In [0]:
%sql
SELECT *
FROM movie_gold.results_movies_production_company

In [0]:
#display(spark.read.parquet(f"{gold_folder_path}/results_movies_production_company"))